In [ ]:
# CoV-36 only × s20_mk2 @ 1M / mrl 64 — CONFIG
# Just the 36 μ-ladder descenders, starting at best_rep (not Aut-min).
# Arm = s20_mk2 = L+20S+2MK. One Colab (no chunking).
# Restart → Run All: UPDATE_REPO + purge picks up runner.

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-u124-s20mk2-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_cov36_s20mk2_1m"

cfg = dict(
    DATASET   = "unsolved124",
    SUBSET    = None,
    COV_ONLY  = True,          # ← only the 36 CoV-reduced best_rep starts

    ARMS      = ["s20_mk2"],

    CHUNKS       = 1,
    CHUNK_INDEX  = 1,

    ENGINE       = "hcompact",
    N_WORKERS    = "auto",
    KEEP_PATH    = False,

    NODE_BUDGET = 1_000_000,
    CHECKPOINTS = [1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000],
    MAX_RELATOR_LENGTH = 64,
    RESUME    = True,
    OUT_STEM  = "hsearch_cov36_s20mk2_covstart_r256b64m24",
    STAGE_DIR = "/content/hsearch_stage",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(cfg.get("STAGE_DIR", "/content/hsearch_stage"), exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("BRANCH:", BRANCH, "ENGINE:", cfg["ENGINE"], "N_WORKERS:", cfg["N_WORKERS"])
assert cfg["ENGINE"] == "hcompact"
assert cfg.get("COV_ONLY") is True

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.runners import run_unsolved124_s20mk2 as _ru
from experiments.heuristic_search.core.hsolve import greedy_search_h
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_ru.run_ab.ARMS["s20_mk2"])
rows = _ru.load_rows(cov_only=True)
print(f"COV_ONLY rows: {len(rows)} (expect 36)")
assert len(rows) == 36 and all(r["cov_reduced"] for r in rows)
a34 = next(r for r in rows if r["name"] == "aca_34")
assert a34["r1"] == "YXXyxYx" and a34["start_total"] == 16
print("aca_34 OK:", a34["r1"], a34["r2"], "names:", [r["name"] for r in rows])
print("kernels warm — setup done")
print("tip: Runtime → Restart → Run All resumes (UPDATE_REPO + flock + RESUME)")


In [ ]:
# ==================== RUN (HIGH_SPEEDUP multi-worker) =====================
from experiments.heuristic_search.runners.run_unsolved124_s20mk2 import (
    run_unsolved124_s20mk2)
from experiments.heuristic_search.runners import run_ab as _ra
nw, per = _ra._resolve_workers(
    {"N_WORKERS": cfg["N_WORKERS"]}, cfg["NODE_BUDGET"],
    cfg["MAX_RELATOR_LENGTH"], cfg["ENGINE"], cfg["KEEP_PATH"])
print(f"HIGH_SPEEDUP resolve: N_WORKERS={cfg['N_WORKERS']} -> {nw} workers "
      f"(~{per:.1f} GB/search est., ENGINE={cfg['ENGINE']})")
print(f"budget={cfg['NODE_BUDGET']:,} mrl={cfg['MAX_RELATOR_LENGTH']} "
      f"COV_ONLY={cfg['COV_ONLY']} arms={cfg['ARMS']}")
run_unsolved124_s20mk2(
    cfg,
    out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else
             "results/heuristic_search/cov36_s20mk2_1m"),
    heartbeat_secs=HEARTBEAT_SECS,
    progress_secs=PROGRESS_SECS,
)
print("done — leave session up until jsonl mirror finishes.")
print("Expect 36 rows; basename contains cov36 + covstart_r256b64m24.")
